# BioImage Archive — Biological Image Data Ingestion and Analysis

**BioImage Archive (BIA)** is a public repository hosted by EMBL-EBI for storing and distributing biological images from life sciences research. It sits within the BioStudies ecosystem and accepts submissions of light microscopy, electron microscopy, X-ray tomography, and other bioimaging modalities.

Key data types:
| Data type | Description |
|---|---|
| **Studies** | Experimental submissions with accessions like `S-BIAD\d+` (BIA-specific) or `S-BSST\d+` |
| **Images** | Raw and processed microscopy images (`.tif`, `.czi`, `.nd2`, `.png`, `.zarr`, etc.) |
| **Annotations** | Segmentation masks, bounding boxes, and label files associated with images |
| **Metadata** | Organism, imaging method, sample preparation, biosample, assay details |

Key fields per study:
| Field | Description |
|---|---|
| `accession` | Unique study identifier, e.g. `S-BIAD144` |
| `title` | Human-readable study title |
| `organism` | Source organism(s) |
| `imaging_method` | Microscopy technique (confocal, CLEM, SEM, etc.) |
| `release_date` | Date the study was made public |
| `file_count` | Number of files deposited |
| `file_size_gb` | Total data volume |

**Reference:** Hartley et al. (2022), *Nature Methods*, BioImage Archive — deposition and retrieval of biological images

**API base:** `https://www.ebi.ac.uk/biostudies/api/v1`

# TODO

* [x] **Ingest data**
    * [x] Connect to BioStudies API and confirm access (fetch study S-BIAD144, print title/organism)
    * [x] Search for BioImage Archive studies and page through results (cache to `data/bia_studies.json`)
    * [x] Parse into a Polars DataFrame: accession, title, organism, imaging_method, release_date, file_count, file_size_gb
    * [x] Fetch the file listing for one study of interest to show image file types and sizes
    * [x] Print shape, dtypes, and head of the studies DataFrame
* [ ] **Explore and clean**
    * [ ] Summarize study counts by organism, imaging method, and release year
    * [ ] Inspect file type distributions across studies
    * [ ] Handle missing/null fields in study metadata
* [ ] **Analysis**
    * [ ] Identify the most-studied organisms and imaging modalities in BIA
    * [ ] Analyse data volume trends over time (cumulative file size by release date)
    * [ ] Cluster studies by keyword/imaging method profile
* [ ] **Visualization**
    * [ ] Bar chart of submissions by organism and year
    * [ ] Heatmap of imaging method × organism usage
    * [ ] File type distribution for a study of interest
* [ ] **Statistical analysis**
    * [ ] Discuss image quality metrics and their statistical basis
    * [ ] Explain multiple hypothesis correction when comparing image-derived phenotypes at scale

In [ ]:
import requests
import time
import json
from pathlib import Path

import polars as pl

## 1. Ingest Data

### 1.1 Connect to BioStudies API and Confirm Access

In [ ]:
BIA_BASE = "https://www.ebi.ac.uk/biostudies/api/v1"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def bia_get(endpoint: str, params: dict = None, headers: dict = None) -> requests.Response:
    """
    Send a GET request to the BioStudies/BioImage Archive REST API.

    Parameters
    ----------
    endpoint : str
        API path relative to BIA_BASE (e.g. "studies/S-BIAD144").
    params : dict, optional
        Query parameters appended to the URL.
    headers : dict, optional
        Extra HTTP headers; ``Accept: application/json`` is always added.

    Returns
    -------
    requests.Response
        Raw response object; caller is responsible for parsing.
    """
    url = f"{BIA_BASE}/{endpoint}"
    hdrs = {"Accept": "application/json"}
    if headers:
        hdrs.update(headers)
    resp = requests.get(url, params=params or {}, headers=hdrs, timeout=30)
    resp.raise_for_status()
    return resp

# Connectivity check: fetch a well-known BIA study
sample = bia_get("studies/S-BIAD144").json()

# The BioStudies API nests metadata under 'attributes' as a list of {name, value} dicts
def get_attr(attrs: list, name: str) -> str | None:
    """Extract the first value matching ``name`` from a BioStudies attributes list."""
    for a in attrs or []:
        if a.get("name", "").lower() == name.lower():
            return a.get("value")
    return None

attrs = sample.get("attributes", [])
print(f"Accession : {sample.get('accno')}")
print(f"Title     : {get_attr(attrs, 'Title')}")
print(f"Organism  : {get_attr(attrs, 'Organism')}")

### 1.2 Search for BioImage Archive Studies and Page Through Results

In [ ]:
STUDIES_CACHE = DATA_DIR / "bia_studies.json"
PAGE_SIZE = 100  # maximum page size the BioStudies search endpoint accepts

def fetch_all_bia_studies(cache_path: Path = STUDIES_CACHE) -> list[dict]:
    """
    Fetch all publicly accessible BioImage Archive studies via the BioStudies
    search endpoint, paging through results until exhausted.  Results are
    cached to disk so subsequent runs load instantly without hitting the API.

    Parameters
    ----------
    cache_path : Path
        File path used to read/write the JSON cache.

    Returns
    -------
    list[dict]
        One dict per study as returned by the BioStudies search API.
    """
    if cache_path.exists():
        print(f"Loading from cache: {cache_path}")
        return json.loads(cache_path.read_text())

    all_studies = []
    page = 1  # BioStudies search is 1-indexed
    while True:
        resp = bia_get(
            "search",
            params={
                "query":    "BioImage",   # filter to BIA-related submissions
                "type":     "study",
                "page":     page,
                "pageSize": PAGE_SIZE,
            },
        )
        data = resp.json()
        batch = data.get("hits", [])
        if not batch:
            break  # no more results

        all_studies.extend(batch)
        total = data.get("totalHits", "?")
        print(f"  Page {page:>4} — {len(all_studies):>6} / {total} studies", end="\r")
        page += 1
        time.sleep(0.3)  # polite delay between paginated calls

    print(f"\nDone. Fetched {len(all_studies)} studies.")
    cache_path.write_text(json.dumps(all_studies))   # persist to disk
    return all_studies

studies_raw = fetch_all_bia_studies()
print(f"Total studies cached: {len(studies_raw)}")

### 1.3 Parse into a Polars DataFrame

In [ ]:
def flatten_study(s: dict) -> dict:
    """
    Flatten a single raw BioStudies search-hit dict into a row-friendly format.

    The search endpoint returns a compact record with top-level scalar fields
    and an ``attributes`` list of ``{name, value}`` pairs for richer metadata.
    We extract the fields most useful for downstream analysis.

    Parameters
    ----------
    s : dict
        Raw study record from the BioStudies ``/search`` endpoint.

    Returns
    -------
    dict
        Flat dict with scalar values ready for a Polars DataFrame row.
    """
    attrs = s.get("attributes", [])  # list of {name, value} dicts

    def attr(name: str) -> str | None:
        """Return first attribute value matching ``name`` (case-insensitive)."""
        for a in attrs:
            if a.get("name", "").lower() == name.lower():
                v = a.get("value")
                return v.strip() if isinstance(v, str) else v
        return None

    # File stats are sometimes provided as top-level keys in search hits
    file_count = s.get("filesCount") or s.get("fileCount")
    file_size  = s.get("filesSize")  or s.get("fileSize")   # bytes, may be None

    return {
        "accession":      s.get("accno") or s.get("accession"),
        "title":          attr("Title"),
        "organism":       attr("Organism"),
        "imaging_method": attr("Imaging method") or attr("Imaging Method"),
        "release_date":   s.get("rtime") or attr("ReleaseDate"),   # Unix ms or date string
        "file_count":     int(file_count) if file_count is not None else None,
        "file_size_gb":   round(int(file_size) / 1e9, 4) if file_size is not None else None,
    }

rows = [flatten_study(s) for s in studies_raw]
studies = pl.DataFrame(rows)

# release_date may be a Unix timestamp in milliseconds (integer) or a date string
# We normalise to a Polars Date regardless of source format
if studies["release_date"].dtype == pl.Utf8:
    studies = studies.with_columns(
        pl.col("release_date").str.to_date(format="%Y-%m-%d", strict=False)
    )
else:
    # Convert millisecond epoch → date (divide by 1000 to get seconds)
    studies = studies.with_columns(
        (pl.col("release_date").cast(pl.Int64, strict=False) // 1000)
        .cast(pl.Datetime)
        .dt.date()
        .alias("release_date")
    )

# Cast numeric columns to correct types
studies = studies.with_columns([
    pl.col("file_count").cast(pl.Int32, strict=False),
    pl.col("file_size_gb").cast(pl.Float64, strict=False),
])

print(f"Shape  : {studies.shape}")
print(f"Memory : {studies.estimated_size('kb'):.1f} KB")
print()
print(studies.dtypes)
print()
studies.head(10)

### 1.4 Fetch File Listing for a Study of Interest

In [ ]:
# S-BIAD144: a publicly available fluorescence microscopy study used in BIA tutorials
FOCUS_STUDY = "S-BIAD144"

def fetch_study_files(accession: str) -> pl.DataFrame:
    """
    Fetch the top-level file directory listing for a BioImage Archive study.

    Uses the BioStudies ``/studies/{accession}/files/directory`` endpoint which
    returns a flat list of file metadata objects including name, size, and path.

    Parameters
    ----------
    accession : str
        BIA study accession, e.g. ``"S-BIAD144"``.

    Returns
    -------
    pl.DataFrame
        One row per file with columns: file_name, extension, size_bytes, size_mb, path.
    """
    resp = bia_get(f"studies/{accession}/files/directory")
    data = resp.json()

    # The endpoint may return a list directly or nest files under a key
    files = data if isinstance(data, list) else data.get("files", [])

    rows = []
    for f in files:
        name = f.get("name") or f.get("fileName") or ""
        size = f.get("size") or f.get("fileSize") or 0
        rows.append({
            "file_name":   name,
            "extension":   Path(name).suffix.lower() or "(none)",  # e.g. ".tif"
            "size_bytes":  int(size),
            "size_mb":     round(int(size) / 1e6, 3),
            "path":        f.get("path") or f.get("filePath") or "",
        })

    return pl.DataFrame(rows).with_columns(
        pl.col("size_bytes").cast(pl.Int64),
        pl.col("size_mb").cast(pl.Float64),
    )

study_files = fetch_study_files(FOCUS_STUDY)
print(f"Files in {FOCUS_STUDY}: {len(study_files)}")
print()

# Summarise by file extension to show the mix of image formats
summary = (
    study_files
    .group_by("extension")
    .agg(
        pl.len().alias("count"),
        (pl.col("size_bytes").sum() / 1e6).round(2).alias("total_mb"),
    )
    .sort("total_mb", descending=True)
)
print(summary)
print()
study_files.head(10)

### 1.5 DataFrame Shape, Dtypes, and Head

In [ ]:
print("=== studies DataFrame ===")
print(f"Shape  : {studies.shape[0]} rows × {studies.shape[1]} columns")
print(f"Memory : {studies.estimated_size('kb'):.1f} KB")
print()
print("Dtypes:")
for col, dtype in zip(studies.columns, studies.dtypes):
    print(f"  {col:<20} {dtype}")
print()
print("Null counts:")
print(studies.null_count())
print()
studies.head(10)